# Project 9 — Data Intelligence over Orders
## Task 3: Full EDA (Exploratory Data Analysis)

**Prerequisite:** Run `Task1_2_Data_Loading_and_SQL.ipynb` first — this notebook expects `olist.db` to already exist in the same folder.

**Quick concept note (not in your sessions):** EDA stands for **Exploratory Data Analysis** — looking at data before modeling it, to understand distributions, patterns, and quality issues so later steps (recommendations, segmentation) are grounded in reality instead of guesswork.

**Goal:** Answer the three core EDA questions from the brief — basket composition, repeat-purchase behavior, reorder timing — using SQL to pull/aggregate, then Pandas/Matplotlib to analyze and visualize.


### Setup — reconnect to the database

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

conn = sqlite3.connect("olist.db")

def q(sql):
    return pd.read_sql_query(sql, conn)

os.makedirs("reports", exist_ok=True)


---
### 3.1 — Basket composition: how many items per order?

In [ ]:
basket_sizes = q("""
SELECT order_id, COUNT(order_item_id) AS num_items
FROM order_items
GROUP BY order_id
""")

print(basket_sizes["num_items"].describe())

plt.figure(figsize=(8,5))
basket_sizes["num_items"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribution of Basket Size (items per order)")
plt.xlabel("Number of items")
plt.ylabel("Number of orders")
plt.savefig("reports/basket_size_distribution.png")
plt.show()


**Look for:** what's the typical basket size? Is it mostly single-item orders, or do multi-item baskets happen often?

---
### 3.2 — Top categories by volume (basket composition, deeper)

In [ ]:
category_pairs = q("""
SELECT ct.product_category_name_english AS category, COUNT(*) AS times_purchased
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY times_purchased DESC
""")

print(category_pairs.head(10))

plt.figure(figsize=(10,6))
category_pairs.head(10).plot(kind="barh", x="category", y="times_purchased", legend=False)
plt.title("Top 10 Categories by Volume")
plt.tight_layout()
plt.savefig("reports/top_categories.png")
plt.show()


---
### 3.3 — Repeat-purchase behavior

In [ ]:
repeat_customers = q("""
SELECT customer_unique_id, COUNT(DISTINCT o.order_id) AS num_orders
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY customer_unique_id
""")

repeat_rate = (repeat_customers["num_orders"] > 1).mean()
print(f"Repeat purchase rate: {repeat_rate:.2%}")
print(repeat_customers["num_orders"].value_counts().sort_index())


**Note:** a low repeat rate is a real, honest finding for Olist-style data — not a bug. Worth stating plainly in your write-up rather than hiding it.

---
### 3.4 — Reorder timing (gap between orders, for repeat customers)

In [ ]:
order_dates = q("""
SELECT c.customer_unique_id, o.order_purchase_timestamp
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
ORDER BY c.customer_unique_id, o.order_purchase_timestamp
""")

order_dates["order_purchase_timestamp"] = pd.to_datetime(order_dates["order_purchase_timestamp"])
order_dates["gap_days"] = order_dates.groupby("customer_unique_id")["order_purchase_timestamp"].diff().dt.days

gaps = order_dates["gap_days"].dropna()
print(gaps.describe())

plt.figure(figsize=(8,5))
gaps.hist(bins=30)
plt.title("Days Between Repeat Orders")
plt.xlabel("Days")
plt.ylabel("Frequency")
plt.savefig("reports/reorder_gap_histogram.png")
plt.show()


**Note:** this step mixes SQL (pulling clean sorted data) with Pandas (`.diff()` for gap calculation) — SQL fetches, Pandas computes what SQL isn't built for. You'll reuse this same gap-calculation pattern later.

---
### 3.5 — Customer lifetime value snapshot

In [ ]:
customer_value = q("""
SELECT c.customer_unique_id,
       COUNT(DISTINCT o.order_id) AS num_orders,
       ROUND(SUM(oi.price), 2) AS total_spent
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
ORDER BY total_spent DESC
""")

print(customer_value.describe())
customer_value.head(10)


---
### 3.6 — Revenue concentration check (80/20 pattern)

In [ ]:
customer_value_sorted = customer_value.sort_values("total_spent", ascending=False).reset_index(drop=True)
customer_value_sorted["cumulative_pct"] = customer_value_sorted["total_spent"].cumsum() / customer_value_sorted["total_spent"].sum()

top_20pct_count = int(len(customer_value_sorted) * 0.2)
top_20pct_revenue_share = customer_value_sorted.iloc[top_20pct_count - 1]["cumulative_pct"]

print(f"Top 20% of customers drive {top_20pct_revenue_share:.1%} of total revenue")


---
### 3.7 — Write up your findings

Summarize 3–5 real findings from the numbers above into `reports/eda_findings.md`, e.g.:
- Average basket size is X items
- Repeat purchase rate is Y%
- Top 20% of customers drive Z% of revenue
- Typical reorder gap is N days

This becomes your Phase 1 EDA write-up and feeds directly into later tasks (segmentation, recommendations).


---
## ✅ Task 3 Complete

You now have a full EDA covering basket composition, repeat-purchase rate, reorder timing, and customer value — with saved charts in `reports/`.

**Next: Task 4 — LangChain SQL agent** (chat with your data), or **Task 5 — Market basket analysis** if you want to prioritize core deliverables first.
